# Phase 5 · LoRA Fine-tuning — Banking77 Intent Classifier

**Model:** `Qwen2.5-0.5B-Instruct`  
**Method:** QLoRA via Unsloth (4-bit base + small trainable adapters)  
**GPU:** Colab free T4 (15 GB VRAM) or Kaggle T4×2  
**Runtime:** ~25 min for 2 epochs on a T4

### Checklist before running
1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Upload `sft_train.jsonl` and `sft_val.jsonl` from your `data/` folder (see Cell 2)
3. Set your Hugging Face token in the Secrets panel (key name: `HF_TOKEN`) — optional, needed to push the adapter

### What this notebook does
```
1. Install Unsloth + dependencies
2. Upload data files
3. Load Qwen2.5-0.5B-Instruct in 4-bit
4. Attach LoRA adapters (rank 16)
5. Train 2 epochs, eval every 250 steps, loss on assistant reply only
6. Plot loss curves
7. Sanity-check on 10 validation examples
8. Save adapter locally + push to HF Hub
```

## Cell 1 · Install Unsloth

In [ ]:
# Unsloth's install command for Colab — takes ~2 min the first time
# If this cell errors, go to Runtime → Disconnect and delete runtime, then try again
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps trl peft accelerate bitsandbytes --quiet
print("Install done.")

## Cell 2 · Upload data files

In [ ]:
from google.colab import files

print("Upload sft_train.jsonl and sft_val.jsonl from your data/ folder.")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

## Cell 3 · Verify data

In [ ]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = load_jsonl("sft_train.jsonl")
val_data   = load_jsonl("sft_val.jsonl")

print(f"Train rows : {len(train_data)}")
print(f"Val rows   : {len(val_data)}")
print()
row = train_data[0]
print("=== Sample row ===")
for m in row["messages"]:
    role = m["role"]
    content = m["content"][:120] + ("..." if len(m["content"]) > 120 else "")
    print(f"[{role}] {content}")

assert len(train_data) > 8000, "Expected ~8997 train rows"
assert len(val_data)   > 900,  "Expected ~1002 val rows"
print("\nData looks good.")

## Cell 4 · Load model in 4-bit with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 576   # from Phase 4 analysis — no truncation needed
DTYPE = None           # Unsloth auto-detects (bfloat16 on A100/L4, float16 on T4)
LOAD_IN_4BIT = True    # QLoRA: 4-bit base model saves ~3× VRAM vs full precision

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print("Model loaded. Parameters:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")

## Cell 5 · Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank — higher = more capacity, more VRAM
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,           # scaling factor: alpha/r = 2 is a common default
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM on long sequences
    random_state=42,
    use_rslora=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of {total:,})")

## Cell 6 · Build HuggingFace datasets (chat format)

In [ ]:
from datasets import Dataset

def apply_chat_template(examples):
    """Apply the model's chat template; returns a list of formatted strings."""
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_ds = Dataset.from_list(train_data).map(apply_chat_template, batched=True)
val_ds   = Dataset.from_list(val_data).map(apply_chat_template, batched=True)

print("Train:", len(train_ds), "Val:", len(val_ds))
print("\nSample formatted text (first 400 chars):")
print(train_ds[0]["text"][:400])

## Cell 7 · Completion-only collator

We compute the loss **only on the assistant reply** (`{"intent": "..."}`).
Without this, ~93% of training signal is wasted re-reading the system prompt
(which the model already knows) instead of learning the classification boundary.

In [ ]:
import torch

# Qwen2.5-Instruct uses <|im_start|>assistant\n as the start-of-reply marker.
# We mask all tokens before this marker so the loss is only on the assistant reply.
# (DataCollatorForCompletionOnlyLM was removed in newer TRL; this does the same thing.)

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
response_token_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)

class CompletionOnlyCollator:
    def __init__(self, tokenizer, response_ids):
        self.tokenizer = tokenizer
        self.response_ids = response_ids

    def __call__(self, features):
        input_ids_list = [torch.tensor(f["input_ids"]) for f in features]
        max_len = max(x.shape[0] for x in input_ids_list)
        pad_id  = self.tokenizer.pad_token_id or 0

        padded   = torch.full((len(features), max_len), pad_id, dtype=torch.long)
        attn     = torch.zeros_like(padded)
        labels   = torch.full_like(padded, -100)

        for i, ids in enumerate(input_ids_list):
            padded[i, :len(ids)] = ids
            attn[i, :len(ids)]   = 1
            seq = ids.tolist()
            # find last occurrence of response template and unmask only the reply
            for j in range(len(seq) - len(self.response_ids), -1, -1):
                if seq[j:j+len(self.response_ids)] == self.response_ids:
                    reply_start = j + len(self.response_ids)
                    labels[i, reply_start:len(ids)] = ids[reply_start:]
                    break

        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

collator = CompletionOnlyCollator(tokenizer, response_token_ids)

# Sanity check: verify the template appears in a formatted example
sample = train_ds[0]["text"]
assert RESPONSE_TEMPLATE in sample, (
    f"Response template not found! Check tokenizer chat template.\n"
    f"Sample snippet: {sample[-200:]}"
)
print(f"Response template found. Token ids: {response_token_ids}")
print("Collator ready.")

## Cell 8 · Training hyperparameters

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

EPOCHS         = 2
BATCH_SIZE     = 4     # per-device; T4 fits 4 comfortably at seq_len 576
GRAD_ACCUM     = 4     # effective batch = 4×4 = 16
LEARNING_RATE  = 2e-4  # standard for LoRA
WARMUP_RATIO   = 0.05
EVAL_STEPS     = 250
SAVE_STEPS     = 250
OUTPUT_DIR     = "banking77-qwen-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=collator,
    args=training_args,
)

steps_per_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Steps per epoch : ~{steps_per_epoch}")
print(f"Total steps     : ~{steps_per_epoch * EPOCHS}")
print(f"Precision       : {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## Cell 9 · Train

In [ ]:
# ~25 min on a T4 for 2 epochs
# Watch: train_loss should drop from ~4+ → ~1 or lower
#        eval_loss should follow (not diverge)
trainer_stats = trainer.train()

print(f"\nTraining done in {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final train loss : {trainer_stats.metrics['train_loss']:.4f}")

## Cell 10 · Plot loss curves

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [e["step"]       for e in log_history if "loss"      in e]
train_losses = [e["loss"]       for e in log_history if "loss"      in e]
eval_steps   = [e["step"]       for e in log_history if "eval_loss" in e]
eval_losses  = [e["eval_loss"]  for e in log_history if "eval_loss" in e]

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_losses, label="train loss",      alpha=0.7)
plt.plot(eval_steps,  eval_losses,  label="validation loss", marker="o", linewidth=2)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Banking77 LoRA — Loss Curves")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()
print("Saved loss_curves.png")

## Cell 11 · Sanity-check on 10 validation examples

If the model is healthy, most of these should be correct **and** valid JSON.
Any invented labels or broken JSON means something went wrong in training.

In [ ]:
import json, random

FastLanguageModel.for_inference(model)  # enable native 2× faster inference

random.seed(0)
sample_rows = random.sample(val_data, 10)

correct = 0
print(f"{'Query':<45} {'True':<35} {'Predicted':<35} {'OK?'}")
print("-" * 125)

for row in sample_rows:
    messages = row["messages"]
    true_intent = json.loads(messages[-1]["content"])["intent"]
    query = messages[1]["content"]

    prompt_messages = messages[:2]  # system + user only
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=30,
            temperature=1.0,
            do_sample=False,
        )

    reply = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    try:
        pred_intent = json.loads(reply)["intent"]
    except Exception:
        pred_intent = f"[BAD JSON] {reply[:40]}"

    ok = pred_intent == true_intent
    if ok:
        correct += 1

    q = (query[:42] + "..") if len(query) > 44 else query
    print(f"{q:<45} {true_intent:<35} {pred_intent:<35} {'✓' if ok else '✗'}")

print(f"\nSanity accuracy on 10 samples: {correct}/10")

## Cell 12 · Save LoRA adapter locally

In [ ]:
ADAPTER_DIR = "banking77-qwen-adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}/")
!ls -lh {ADAPTER_DIR}/

## Cell 13 · (Optional) Push adapter to Hugging Face Hub

Requires `HF_TOKEN` in Colab Secrets (key icon in the left sidebar).
Skip this cell if you just want the local files.

In [ ]:
import os
from google.colab import userdata

HF_USERNAME = "muzammilahmad01"   # your HF username
REPO_NAME   = "banking77-qwen2.5-0.5b-lora"

try:
    hf_token = userdata.get("HF_TOKEN")
    model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=hf_token)
    tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=hf_token)
    print(f"Pushed to https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")
except Exception as e:
    print(f"Push skipped: {e}")
    print("Download the adapter folder manually (Files panel on the left).")

## Cell 14 · Download the adapter and loss curve to your Mac

Download only the small LoRA adapter (~35 MB) and do the merge + GGUF export on your Mac.

> **Why not export GGUF here?** Merging and quantizing inside Colab ran out of memory,
> crashed the runtime, and lost the trained model. The adapter is all you need:
> `backend/merge_adapter.py` merges it locally (see Phase 6 in `plan.md`).


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}.zip

files.download(f"{ADAPTER_DIR}.zip")   # unzip into models/banking77-qwen-adapter/ in the repo
files.download("loss_curves.png")      # save to results/loss_curves_lora.png


---
## What comes next (Phase 6, on your Mac)

1. Unzip the adapter into `models/banking77-qwen-adapter/`
2. `python backend/merge_adapter.py`: merge into the full-precision base and write the `Modelfile`
3. Convert and quantize with llama.cpp (`convert_hf_to_gguf.py` → `llama-quantize … Q4_K_M`)
4. `ollama create banking77-qwen -f Modelfile`
5. Evaluate: `python backend/eval_baseline.py --model banking77-qwen`
